# TSE 2026 — Laboratório forense / Fase 5R (opção W)

## A diferença dentro das zonas é específica das 20h04?

**Resultado anterior V:** em 1.380 zonas com pelo menos 1.000 votos válidos de cada lado do corte original, o grupo posterior teve participação média ponderada de Lula **1,9928 ponto percentual maior** que o grupo anterior. Isso não estabelece a causa.

A opção W repete **o mesmo protocolo descritivo** em dez horários fixados previamente, incluindo o corte principal **04/10/2026 20:04:39 BRT**. As coortes de zonas elegíveis podem mudar com o corte: mostramos cobertura e sinal explicitamente. Um efeito persistente em horários anteriores indicaria **tendência temporal de composição**, não uma mudança exclusiva na retomada nacional.

**Escopo:** BWEB de primeiro turno, timestamps de recebimento por segundo; não equivalem a publicação EA20. Dados oficiais H/M já validados; nenhum novo download, ZIP, SQLite nem votos do segundo turno.

## 1. Fontes verificadas

A opção W cria três CSVs e um sumário JSON. Cada CSV deve coincidir com o SHA-256 registrado no sumário. O programa W reconcilia os contadores da opção V antes de gerar os resultados.

In [ ]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def repo_dir():
    for p in (Path.cwd(),*Path.cwd().parents):
        if (p/"start_windows.bat").is_file():
            return p
    raise FileNotFoundError("Notebook fora do projeto TSE.")

REPO=repo_dir()
sys.path.insert(0,str(REPO))
from urnas import root_path
BASE=root_path(None)/"analysis"/"periodicidade_bweb_2026"

def sha256(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for block in iter(lambda:f.read(4*1024*1024),b""):
            h.update(block)
    return h.hexdigest()

wpath=BASE/"phase5r_cutoff_placebos_2026_summary.json"
vpath=BASE/"phase5q_zone_holdout_2026_summary.json"
w=json.loads(wpath.read_text(encoding="utf-8"))
v=json.loads(vpath.read_text(encoding="utf-8"))
assert w["status"]=="PLACEBOS_TEMPORAIS_INTRA_ZONA_2026"
assert w["year"]==v["year"]==2026 and w["round"]==v["round"]==1
assert w["total_valid"]==119300788

def verified(meta):
    f=Path(meta["path"])
    if not f.is_file():
        f=BASE/f.name
    if sha256(f)!=meta["sha256"]:
        raise ValueError("SHA-256 do arquivo nao confere: "+str(f))
    return pd.read_csv(f,sep=";",encoding="utf-8-sig",
                       dtype={"UF":str,"municipio":str,"zona":str})

cuts=verified(w["output_files"]["cutoffs"])
ufs=verified(w["output_files"]["ufs"])
zones=verified(w["output_files"]["zones"])
cuts["data_hora"]=pd.to_datetime(cuts["cutoff_brt"])
assert w["focus"]["paired_zones"]==v["paired_zones_with_1000_pre_and_post"]
assert abs(w["focus"]["weighted_shift_lula_pp"]-
           v["paired_zones_weighted_delta_post_minus_pre_lula_pp"])<1e-8
print("SHA verificado; cortes:",len(cuts),"zonas elegiveis no corte principal:",len(zones))
display(cuts[["cutoff_brt","all_pre_valid","all_post_valid","paired_zones",
              "paired_post_fraction","weighted_shift_lula_pp","positive_zone_fraction"]].round(4))

## 2. A curva de placebos temporais

Uma mudança **específica do corte** precisaria diferir de forma robusta do comportamento em horários próximos. Observe a direção e magnitude do deslocamento Lula pós−pré, e o tamanho do grupo de votos elegíveis em cada horário. Não atribuímos valor-p a uma tendência observacional.

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
ax.plot(cuts["data_hora"],cuts["weighted_shift_lula_pp"],marker="o",label="Lula, pos - pre (mesmas zonas)")
ax.plot(cuts["data_hora"],cuts["weighted_shift_flavio_pp"],marker="s",label="Flavio, pos - pre (mesmas zonas)")
ax.axvline(pd.Timestamp(w["cutoff_primary"]),linestyle="--",label="Corte original 20:04:39")
ax.axhline(0,linestyle=":",alpha=.7)
ax.set(title="Placebos temporais: mudança de composição dentro das zonas",
       ylabel="Diferença ponderada por votos posteriores (pp)",xlabel="Corte de DT_BU_RECEBIDO")
ax.grid(alpha=.2)
ax.legend()
fig.autofmt_xdate()
plt.show()

## 3. Cobertura e representatividade

Mudanças bruscas podem refletir entrada/saída de zonas do filtro de **mínimo 1.000 votos válidos antes E depois**. Essa cobertura precisa ser analisada junto ao gráfico anterior.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(cuts["data_hora"],cuts["paired_zones"],marker="o",label="Zonas elegíveis")
ax.set(xlabel="Horário do corte",ylabel="Zonas com >=1.000 votos de cada lado",
       title="Cobertura de zonas pareadas nos placebos")
ax.grid(alpha=.2)
fig.autofmt_xdate()
plt.show()
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(cuts["data_hora"],100*cuts["paired_post_fraction"],marker="o")
ax.set(xlabel="Horário do corte",ylabel="% dos votos pós-corte cobertos",
       title="Porcentagem da cauda representada pelas zonas elegíveis")
ax.grid(alpha=.2)
fig.autofmt_xdate()
plt.show()

## 4. O efeito aparece na maioria das zonas ou em poucas zonas grandes?

A parcela de zonas com aumento para Lula é calculada apenas sobre as **zonas elegíveis naquele corte**. A média ponderada pode ser dominada por um número pequeno de zonas; o sinal da maioria ajuda a identificar concentração.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(cuts["data_hora"],100*cuts["positive_zone_fraction"],marker="o")
ax.axhline(50,linestyle="--")
ax.set(xlabel="Horário de corte",ylabel="% de zonas com aumento para Lula",
       title="Distribuição do sinal dos deslocamentos dentro das zonas")
ax.set_ylim(0,100)
ax.grid(alpha=.2)
fig.autofmt_xdate()
plt.show()
display(cuts[["cutoff_brt","paired_zones","positive_zone_count",
              "negative_zone_count","weighted_shift_lula_pp",
              "unweighted_median_shift_lula_pp"]].round(3))

## 5. Robustez geográfica do corte principal

A opção W também recalcula a média ponderada **retirando uma UF de cada vez**. Isso testa se a mudança média nacional depende excessivamente de uma única UF. Não equivale a experimento randomizado nem elimina possíveis padrões de recebimento comuns a todos os estados.

In [ ]:
leave=pd.DataFrame(w["robustness"]["leave_one_UF_out"])
focus=w["focus"]
display(leave.sort_values("weighted_shift_lula_pp").round(4))
print("Média ponderada nacional:",round(focus["weighted_shift_lula_pp"],5),"pp")
print("Mínimo sem uma UF:",round(leave["weighted_shift_lula_pp"].min(),5),"pp")
print("Máximo sem uma UF:",round(leave["weighted_shift_lula_pp"].max(),5),"pp")
est=ufs.sort_values("paired_post_valid",ascending=False).copy()
display(est[["UF","paired_zones","paired_pre_valid","paired_post_valid",
             "weighted_shift_lula_pp","positive_zones"]].head(27).round(3))

## 6. Zonas extremas, ordenadas pelo volume (sem nomes inventados)

As zonas abaixo tiveram muitos votos dos dois lados e mudança percentual. São **alvos de auditoria descritiva**, não identificação de votos adulterados. Para investigar uma zona específica, associar município e zona aos registros oficiais sem inferir identidade da seção pelo código numérico isolado.

In [ ]:
display(zones.sort_values("post_valid",ascending=False)
        [["UF","municipio","zona","pre_valid","post_valid",
          "pre_lula_pct","post_lula_pct","shift_lula_pp","shift_flavio_pp"]]
        .head(30).round(3))
fig,ax=plt.subplots(figsize=(8,6))
ax.scatter(zones["pre_lula_pct"],zones["post_lula_pct"],
           s=7+40*zones["post_valid"]/zones["post_valid"].max(),alpha=.4)
ax.plot([0,100],[0,100],linestyle="--")
ax.set(xlabel="Lula pré-corte (%)",ylabel="Lula pós-corte (%)",
       title="Zonas elegíveis do corte principal, antes versus depois")
ax.set_xlim(0,100)
ax.set_ylim(0,100)
ax.grid(alpha=.2)
plt.show()

## 7. Interpretação investigativa

- Se o efeito dentro da zona cresce gradualmente em todos os placebos, isso apoia uma explicação de **composição temporal não aleatória** dos BUs; não explica automaticamente a forma da publicação EA20.
- Se houver salto excepcional apenas perto das 20h04, ele merece investigação operacional com dados de lote, timestamps e cadeia de publicação.
- Se a mudança se concentrar em poucas UFs ou zonas, o próximo passo é reconstruir os BUs individuais desses territórios e examinar eventuais diferenças entre seções.
- Sem controle sobre **como** as seções entraram nas filas de recebimento, o teste não permite inferir manipulação de votos a partir de um desvio estatístico.

**Pendência principal:** explicar por que a curva publicada EA20 tem R² = 0,99044 contra o número cumulativo de seções. O experimento W testa recebimento BWEB e não substitui a auditoria de buffering e publicação.

In [ ]:
print("Fase:",w["version"])
print("Cortes fixos:",len(w["cutoffs"]))
print("Corte principal:",w["cutoff_primary"])
print("Zonas pareadas:",w["focus"]["paired_zones"])
print("Votos pós-corte pareados:",w["focus"]["paired_post_valid"])
print("Lula pos - pre, ponderado:",w["focus"]["weighted_shift_lula_pp"])
print("SHA do relatorio de cortes:",w["output_files"]["cutoffs"]["sha256"])
print("Fontes originais inalteradas.")